# Simple Linear Regression vs Multiple Linear Regression

This notebook demonstrates how to build and compare a simple linear regression model and a multiple linear regression model using synthetic data, scikit-learn, and Python.

We will:
- generate a reproducible dataset with 5,000 observations
- save it as a CSV file
- train a Simple Linear Regression (SLR) model
- train a Multiple Linear Regression (MLR) model
- evaluate both models using MSE, MAE, and R²
- visualize 10 test-set predictions and residuals

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

# Use a fixed random seed so the dataset is reproducible
rng = np.random.default_rng(42)
n_samples = 5000

feature_1 = rng.normal(50, 10, n_samples)
feature_2 = rng.normal(30, 8, n_samples)
feature_3 = rng.normal(100, 15, n_samples)
feature_4 = rng.normal(5, 2, n_samples)
feature_5 = rng.normal(0, 1, n_samples)

# Target depends on the input features, with some random noise
target = (
    15
    + 3.2 * feature_1
    - 2.5 * feature_2
    + 0.9 * feature_3
    + 4.7 * feature_4
    + 1.5 * feature_5
    + rng.normal(0, 5, n_samples)
)

df = pd.DataFrame({
    'feature_1': feature_1,
    'feature_2': feature_2,
    'feature_3': feature_3,
    'feature_4': feature_4,
    'feature_5': feature_5,
    'target': target
})

data_path = Path.cwd() / 'regression_data.csv'
df.to_csv(data_path, index=False)
print(f'Dataset saved to: {data_path}')
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
data_path = Path.cwd() / 'regression_data.csv'
df = pd.read_csv(data_path)

print('Dataset shape:', df.shape)
display(df.head())

## Simple Linear Regression vs Multiple Linear Regression

A simple linear regression model uses only one input feature to predict the target. In this example, we will use `feature_1` as the single predictor.

A multiple linear regression model uses more than one input feature. Here, we will use several features such as `feature_1`, `feature_2`, `feature_3`, and `feature_4` to predict the same target.

The main difference is that simple linear regression fits a single straight line, while multiple linear regression fits a higher-dimensional linear relationship.

## Model evaluation metrics

We will evaluate each model with the following metrics:

- Mean Squared Error (MSE): measures the average squared difference between the actual and predicted values. Smaller values indicate better predictions.
- Mean Absolute Error (MAE): measures the average absolute difference between actual and predicted values. It is easier to interpret because it is in the same units as the target.
- R² score: tells us how much of the variation in the target is explained by the model. A value closer to 1 means the model fits the data better.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Choose the target
y = df['target']

# SLR uses one feature
X_slr = df[['feature_1']]

# MLR uses multiple features
X_mlr = df[['feature_1', 'feature_2', 'feature_3', 'feature_4']]

# Split the data into training and testing sets
X_slr_train, X_slr_test, y_slr_train, y_slr_test = train_test_split(
    X_slr, y, test_size=0.2, random_state=42
)
X_mlr_train, X_mlr_test, y_mlr_train, y_mlr_test = train_test_split(
    X_mlr, y, test_size=0.2, random_state=42
)

# Train the models
slr_model = LinearRegression()
mlr_model = LinearRegression()

slr_model.fit(X_slr_train, y_slr_train)
mlr_model.fit(X_mlr_train, y_mlr_train)

# Make predictions on the test set
slr_pred = slr_model.predict(X_slr_test)
mlr_pred = mlr_model.predict(X_mlr_test)

print('SLR model trained successfully.')
print('MLR model trained successfully.')

In [ ]:
def summarize_model(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'\n{name} Results')
    print('-' * 30)
    print(f'MSE:  {mse:.4f}')
    print(f'MAE:  {mae:.4f}')
    print(f'R²:   {r2:.4f}')
    return {'MSE': mse, 'MAE': mae, 'R²': r2}

slr_metrics = summarize_model('Simple Linear Regression', y_slr_test, slr_pred)
mlr_metrics = summarize_model('Multiple Linear Regression', y_mlr_test, mlr_pred)

## Interpreting residuals in the plots

Each point in the prediction plot shows one actual value and one predicted value. The residual is calculated as:

 residual = actual - predicted

A positive residual means the model under-predicted the true value, because the actual value is greater than the prediction. A negative residual means the model over-predicted the true value, because the prediction is greater than the actual value.

The size of the residual tells us how far off the prediction was. Larger absolute residuals indicate weaker predictions, while residuals close to zero suggest the model is performing well.

In [ ]:
import matplotlib.pyplot as plt

# Pick 10 sample observations for a clean comparison plot
sample_indices = np.linspace(0, len(y_slr_test) - 1, 10, dtype=int)
sample_actual = y_slr_test.iloc[sample_indices].to_numpy()
sample_pred = slr_pred[sample_indices]
sample_residual = sample_actual - sample_pred

plt.figure(figsize=(10, 6))
x_vals = np.arange(len(sample_actual))
plt.plot(x_vals, sample_actual, marker='o', linestyle='-', linewidth=2, label='Actual')
plt.plot(x_vals, sample_pred, marker='s', linestyle='--', linewidth=2, label='Predicted')

for i in range(len(sample_actual)):
    plt.plot([x_vals[i], x_vals[i]], [sample_actual[i], sample_pred[i]],
             color='gray', linestyle=':', linewidth=1.2, alpha=0.8)
    plt.text(x_vals[i], min(sample_actual[i], sample_pred[i]) - 2.5,
             f'{sample_residual[i]:.2f}', ha='center', va='top', fontsize=8, color='black')

plt.title('SLR: Actual vs Predicted Values on 10 Test Observations')
plt.xlabel('Sample index')
plt.ylabel('Target value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
sample_indices = np.linspace(0, len(y_mlr_test) - 1, 10, dtype=int)
sample_actual = y_mlr_test.iloc[sample_indices].to_numpy()
sample_pred = mlr_pred[sample_indices]
sample_residual = sample_actual - sample_pred

plt.figure(figsize=(10, 6))
x_vals = np.arange(len(sample_actual))
plt.plot(x_vals, sample_actual, marker='o', linestyle='-', linewidth=2, label='Actual')
plt.plot(x_vals, sample_pred, marker='s', linestyle='--', linewidth=2, label='Predicted')

for i in range(len(sample_actual)):
    plt.plot([x_vals[i], x_vals[i]], [sample_actual[i], sample_pred[i]],
             color='gray', linestyle=':', linewidth=1.2, alpha=0.8)
    plt.text(x_vals[i], min(sample_actual[i], sample_pred[i]) - 2.5,
             f'{sample_residual[i]:.2f}', ha='center', va='top', fontsize=8, color='black')

plt.title('MLR: Actual vs Predicted Values on 10 Test Observations')
plt.xlabel('Sample index')
plt.ylabel('Target value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Final thoughts

In this synthetic example, the multiple regression model usually performs better because it uses more relevant information from several features. The simple linear regression model is easier to interpret but may not capture the full relationship in the data.

By comparing MSE, MAE, and R² and by examining the residuals visually, we can judge how well each model is learning the pattern in the target variable.